# Bronze Layer - Raw Ingestion
Raw CSV → Delta, AS-IS. No type casting, no filtering, no dedup.
Only added: `ingested_at` audit metadata. Parsing deferred to Silver.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = f"{CATALOG}.default"

sessions_path = "/Volumes/workspace/default/ev_data/sessions.csv"
stations_path = "/Volumes/workspace/default/ev_data/stations.csv"

# Sessions — AS-IS, all strings, audit timestamp only
bronze_sessions = (
    spark.read.csv(sessions_path, header=True, inferSchema=False)
    .withColumn("ingested_at", F.current_timestamp())
)
bronze_sessions.write.mode("overwrite").saveAsTable(f"{SCHEMA}.bronze_sessions")
print(f"bronze_sessions: {bronze_sessions.count()} rows")

# Stations — same rule, AS-IS
bronze_stations = (
    spark.read.csv("/Volumes/workspace/default/ev_data/stations.csv",
                   header=True, inferSchema=False)
    .withColumn("ingested_at", F.current_timestamp())
)
bronze_stations.write.mode("overwrite").saveAsTable(f"{SCHEMA}.bronze_stations")
print(f"Bronze stations: {bronze_stations.count()}")